# RuVLM BenchMax — обучение на GPU (Google Colab)

Дообучение `deepvk/llava-gemma-2b-lora` на открытых данных VK (GQA-ru) через LoRA.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**.

Если была ошибка RAM: **Runtime → Disconnect and delete runtime**, затем заново.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Включите GPU в Runtime → Change runtime type"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Стабильный набор для Colab (bf16 LoRA, без bitsandbytes/torchao).
!pip uninstall -y torchao bitsandbytes -q 2>/dev/null || true
!pip install -q --upgrade --force-reinstall "pillow==11.1.0"
!pip install -q "transformers==4.46.3" "peft==0.13.2" "trl==0.12.2" "datasets==3.2.0" "accelerate==1.2.1" PyYAML

print("deps ready — Runtime → Restart session, затем ячейки A → B")

## LoRA fine-tune (экономно по RAM)

На обычном Colab (~12 GB RAM) нельзя держать модель 3B **и** все 27k картинок сразу.

1. **A** — только маленький subset данных  
2. **B** — модель + обучение  

При OOM: Restart session и поставьте `MAX_SAMPLES = 128`.

In [ ]:
# === A. Данные (модель ещё НЕ загружаем) ===
import gc
from datasets import Dataset, load_dataset

MAX_SAMPLES = 256  # при OOM снизьте до 128
POST_PROMPT = " Ответь одним словом."
SEED = 42

instr = load_dataset("deepvk/GQA-ru", "train_balanced_instructions", split="train")
instr = instr.shuffle(seed=SEED).select(range(min(MAX_SAMPLES, len(instr))))
needed = list(dict.fromkeys(instr["imageId"]))

# НЕ используем .filter() по всем 27k — это съедает RAM.
# Берём только индексы нужных картинок.
imgs = load_dataset("deepvk/GQA-ru", "train_balanced_images", split="train")
id2idx = {iid: i for i, iid in enumerate(imgs["id"])}
indices = [id2idx[i] for i in needed if i in id2idx]
imgs_small = imgs.select(indices)

img_by_id = {}
for row in imgs_small:
    img_by_id[row["id"]] = row["image"].convert("RGB")

del imgs, imgs_small, id2idx, indices
gc.collect()

rows = []
for ex in instr:
    q = ex["question"].rstrip() + POST_PROMPT
    a = ex.get("answer") or ex.get("fullAnswer") or ""
    rows.append(
        {
            "messages": [
                {"role": "user", "content": f"<image>\n{q}"},
                {"role": "assistant", "content": str(a)},
            ],
            "image": img_by_id[ex["imageId"]],
        }
    )

train_ds = Dataset.from_list(rows)
del instr, rows, img_by_id, needed
gc.collect()
print(f"ready: {len(train_ds)} samples")

In [ ]:
# === B. Модель + обучение ===
import gc
import torch
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoProcessor,
    AutoTokenizer,
    LlavaForConditionalGeneration,
    TrainingArguments,
)
from trl import SFTTrainer

MODEL_ID = "deepvk/llava-gemma-2b-lora"
OUTPUT_DIR = "outputs/ruvlm-gemma-2b-lora"
NUM_EPOCHS = 1

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,  # fp16 экономит RAM vs bf16 на T4
    low_cpu_mem_usage=True,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
processor.tokenizer = tokenizer

lora = LoraConfig(
    r=16,  # меньше r → меньшеtrainable params / RAM
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
model.enable_input_require_grads()


class Collator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        texts, images = [], []
        for ex in examples:
            text = self.processor.tokenizer.apply_chat_template(
                ex["messages"], tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            images.append(ex["image"])
        batch = self.processor(text=texts, images=images, return_tensors="pt", padding=True)
        labels = batch["input_ids"].clone()
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100
        batch["labels"] = labels
        return batch


args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    logging_steps=5,
    save_steps=500,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    optim="adamw_torch",
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to="none",
    max_grad_norm=1.0,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=Collator(processor),
    tokenizer=tokenizer,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
gc.collect()
torch.cuda.empty_cache()
print("Saved to", OUTPUT_DIR)

## Демо инференса

Запускайте **после** обучения (не параллельно с ячейкой A).

In [ ]:
import requests
from PIL import Image

url = "https://www.ilankelman.org/stopsigns/australia.jpg"
img = Image.open(requests.get(url, stream=True).raw).convert("RGB")
messages = [{"role": "user", "content": "<image>\nОпиши картинку несколькими словами."}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(images=[img], text=text, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Оценка на бенчмарках

После сохранения чекпоинта (лучше на GPU с большим объёмом RAM):

```bash
accelerate launch -m lmms_eval --model llava_hf \
  --model_args pretrained=outputs/ruvlm-gemma-2b-lora \
  --tasks gqa-ru,mmbench_ru_dev --batch_size 1 \
  --output_path ./logs/
```

Запишите метрики в `results/metrics.md` и `docs/MODEL_CARD.md`.